# P2b · Headroom от холодных юзеров (trade + genre)

**Вопрос:** насколько вырастет **общий** NDCG@10, если холодные юзеры будут предсказываться так же хорошо, как тёплые?

**Предиктор:** moving-average по всей прошлой истории юзера (распределение интереса = среднее его прошлых дней). Почему именно он:
- это ровно эвристика, которую обучаемая модель аппроксимирует, а TGN структурно представить не может (Thm 1 статьи);
- его качество естественно зависит от «тёплости» (числа прошлых дней) → можно мерить headroom по бинам тёплости;
- считается быстро на обоих датасетах без обучения (TGNv2 на genre на CPU ≈ 26 мин/эпоха — нереально в ноутбуке).

**Связь с моделью:** в P2 TGNv2 на trade дал ту же деградацию холодные→тёплые (0.378 → 0.641), так что headroom реален и для обучаемой модели; плотная per-(user, item) память метит именно в холодную когорту.

**Тёплость** = число прошлых активных дней юзера. Конвенции: Polars, Plotly, русский.

In [1]:
import polars as pl, numpy as np, plotly.express as px
from sklearn.metrics import ndcg_score

DS = "/Users/aleksandrpanysev/miniconda3/envs/tgb/lib/python3.13/site-packages/tgb/datasets"
COLS = {  # (ts, user, item) — имена колонок различаются по датасетам
    "tgbn-trade": ("year", "nation", "trading nation"),
    "tgbn-genre": ("ts", "user_id", "genre"),
}

def load_norm(name):
    ts, u, it = COLS[name]
    lab = (pl.read_csv(f"{DS}/{name.replace('-', '_')}/{name}_node_labels.csv")
             .select([pl.col(ts).alias("ts"), pl.col(u).alias("user"),
                      pl.col(it).alias("item"), pl.col("weight")]))
    wsum = lab.group_by(["ts", "user"]).agg(pl.col("weight").sum().alias("wsum"))
    return lab.join(wsum, on=["ts", "user"]).with_columns((pl.col("weight") / pl.col("wsum")).alias("p"))

def ma_ndcg_by_warmth(name, n_sample=10000, k=10, seed=1):
    """Per-(user, день) NDCG@10 предиктора MA-по-всей-истории + тёплость (#прошлых дней)."""
    norm = load_norm(name)
    # тёплость = 0-based индекс дня юзера
    uts = norm.select(["user", "ts"]).unique().sort(["user", "ts"]) \
              .with_columns(pl.int_range(pl.len()).over("user").alias("warmth"))
    # MA-числитель: накопленная масса по (user, item) ДО текущего дня
    norm = norm.sort(["user", "item", "ts"]).with_columns(
        (pl.col("p").cum_sum().over(["user", "item"]) - pl.col("p")).alias("prior_cum"))
    norm = norm.join(uts, on=["user", "ts"])
    ma = (norm.filter((pl.col("warmth") >= 1) & (pl.col("prior_cum") > 0))
              .with_columns((pl.col("prior_cum") / pl.col("warmth")).alias("ma")))

    items = norm["item"].unique().to_list()
    G = len(items)
    imap = pl.DataFrame({"item": items, "ii": list(range(G))})

    pairs = uts.filter(pl.col("warmth") >= 1)
    samp = pairs.sample(n=min(n_sample, pairs.height), seed=seed).with_row_index("sid")

    def dense(df, val):
        a = np.zeros((samp.height, G)); a[df["sid"].to_numpy(), df["ii"].to_numpy()] = df[val].to_numpy(); return a

    Yt = dense(samp.join(norm.select(["user", "ts", "item", "p"]), on=["user", "ts"]).join(imap, on="item"), "p")
    Pp = dense(samp.join(ma.select(["user", "ts", "item", "ma"]), on=["user", "ts"]).join(imap, on="item"), "ma")
    keep = np.where((Yt.sum(1) > 0) & (Pp.sum(1) > 0))[0]
    nd = np.array([ndcg_score(Yt[i:i+1], Pp[i:i+1], k=k) for i in keep])
    return samp[keep].select("warmth").with_columns(pl.Series("ndcg", nd))

# быстрая проверка на trade
_t = ma_ndcg_by_warmth("tgbn-trade")
print(f"trade: точек={_t.height}, MA NDCG@10={_t['ndcg'].mean():.3f}, "
      f"тёплость min/med/max={_t['warmth'].min()}/{_t['warmth'].median():.0f}/{_t['warmth'].max()}")

trade: точек=6208, MA NDCG@10=0.828, тёплость min/med/max=1/14/29


In [2]:
def headroom(df, name, nbins=5):
    """Бинуем по тёплости, считаем uplift: что если все выйдут на уровень тёплых."""
    d = df.with_columns(pl.col("warmth").qcut(nbins, allow_duplicates=True).alias("bin"))
    tab = (d.group_by("bin").agg(pl.col("warmth").mean().round(1).alias("avg_warmth"),
                                 pl.col("ndcg").mean().round(3).alias("ndcg"),
                                 pl.len().alias("n"))
             .sort("avg_warmth"))
    W = tab["ndcg"][-1]                       # уровень самой тёплой когорты
    cur = df["ndcg"].mean()
    oracle = df.select(pl.max_horizontal(pl.col("ndcg"), pl.lit(W)))["ndcg"].mean()  # все ≥ W
    print(f"=== {name} ===")
    print(tab)
    print(f"тёплая когорта W = {W:.3f} | текущий общий NDCG@10 = {cur:.3f}")
    print(f"oracle (все холодные → уровень тёплых) = {oracle:.3f}")
    print(f"UPLIFT = {oracle - cur:+.3f}  ({(oracle/cur - 1)*100:.1f}% относительно; "
          f"закрывает {(oracle - cur)/(1 - cur)*100:.0f}% оставшегося до 1.0)\n")
    fig = px.bar(tab.with_columns(pl.col("avg_warmth").cast(pl.Utf8)).to_pandas(),
                 x="avg_warmth", y="ndcg", title=f"{name}: MA NDCG@10 vs тёплость юзера",
                 labels={"avg_warmth": "средняя тёплость (прошлых дней)"})
    fig.add_hline(y=W, line_dash="dash", annotation_text=f"уровень тёплых = {W:.3f}")
    fig.show()
    return {"name": name, "current": cur, "W": W, "oracle": oracle, "uplift": oracle - cur}

res_trade = headroom(_t, "tgbn-trade")

=== tgbn-trade ===
shape: (5, 4)
┌───────────┬────────────┬───────┬──────┐
│ bin       ┆ avg_warmth ┆ ndcg  ┆ n    │
│ ---       ┆ ---        ┆ ---   ┆ ---  │
│ cat       ┆ f64        ┆ f64   ┆ u32  │
╞═══════════╪════════════╪═══════╪══════╡
│ (-inf, 6] ┆ 3.5        ┆ 0.847 ┆ 1425 │
│ (6, 11]   ┆ 9.0        ┆ 0.836 ┆ 1159 │
│ (11, 17]  ┆ 14.5       ┆ 0.821 ┆ 1302 │
│ (17, 23]  ┆ 20.5       ┆ 0.82  ┆ 1238 │
│ (23, inf] ┆ 26.4       ┆ 0.811 ┆ 1084 │
└───────────┴────────────┴───────┴──────┘
тёплая когорта W = 0.811 | текущий общий NDCG@10 = 0.828
oracle (все холодные → уровень тёплых) = 0.895
UPLIFT = +0.067  (8.1% относительно; закрывает 39% оставшегося до 1.0)



In [3]:
_g = ma_ndcg_by_warmth("tgbn-genre", n_sample=12000)
print(f"genre: точек={_g.height}, MA NDCG@10={_g['ndcg'].mean():.3f}, "
      f"тёплость min/med/max={_g['warmth'].min()}/{_g['warmth'].median():.0f}/{_g['warmth'].max()}\n")
res_genre = headroom(_g, "tgbn-genre")

genre: точек=11943, MA NDCG@10=0.847, тёплость min/med/max=1/199/1361

=== tgbn-genre ===
shape: (5, 4)
┌────────────┬────────────┬───────┬──────┐
│ bin        ┆ avg_warmth ┆ ndcg  ┆ n    │
│ ---        ┆ ---        ┆ ---   ┆ ---  │
│ cat        ┆ f64        ┆ f64   ┆ u32  │
╞════════════╪════════════╪═══════╪══════╡
│ (-inf, 65] ┆ 32.2       ┆ 0.835 ┆ 2410 │
│ (65, 148]  ┆ 105.3      ┆ 0.854 ┆ 2386 │
│ (148, 261] ┆ 201.6      ┆ 0.852 ┆ 2380 │
│ (261, 446] ┆ 343.6      ┆ 0.849 ┆ 2393 │
│ (446, inf] ┆ 661.7      ┆ 0.842 ┆ 2374 │
└────────────┴────────────┴───────┴──────┘
тёплая когорта W = 0.842 | текущий общий NDCG@10 = 0.847
oracle (все холодные → уровень тёплых) = 0.905
UPLIFT = +0.058  (6.9% относительно; закрывает 38% оставшегося до 1.0)



In [ ]:
# Диагностика: genre MA NDCG@10=0.847 vs статья 0.509 — почему?
def ma_eval_region(name, lo=0.0, hi=1.0, n=12000, k=10, seed=1):
    norm = load_norm(name)
    uts = (norm.select(["user", "ts"]).unique().sort(["user", "ts"])
              .with_columns(pl.int_range(pl.len()).over("user").alias("warmth")))
    norm = (norm.sort(["user", "item", "ts"])
                .with_columns((pl.col("p").cum_sum().over(["user", "item"]) - pl.col("p")).alias("prior_cum"))
                .join(uts, on=["user", "ts"]))
    ma = (norm.filter((pl.col("warmth") >= 1) & (pl.col("prior_cum") > 0))
              .with_columns((pl.col("prior_cum") / pl.col("warmth")).alias("ma")))
    items = norm["item"].unique().to_list(); G = len(items)
    imap = pl.DataFrame({"item": items, "ii": list(range(G))})
    tsu = norm["ts"].unique().sort(); n_ts = len(tsu)
    lt = tsu[int(n_ts * lo)]; ht = tsu[int(n_ts * hi)] if int(n_ts * hi) < n_ts else tsu[-1] + 1
    pairs = uts.filter((pl.col("warmth") >= 1) & (pl.col("ts") >= lt) & (pl.col("ts") < ht))
    samp = pairs.sample(n=min(n, pairs.height), seed=seed).with_row_index("sid")
    def dense(df, val):
        a = np.zeros((samp.height, G)); a[df["sid"].to_numpy(), df["ii"].to_numpy()] = df[val].to_numpy(); return a
    Yt = dense(samp.join(norm.select(["user", "ts", "item", "p"]), on=["user", "ts"]).join(imap, on="item"), "p")
    Pp = dense(samp.join(ma.select(["user", "ts", "item", "ma"]), on=["user", "ts"]).join(imap, on="item"), "ma")
    keep = np.where((Yt.sum(1) > 0) & (Pp.sum(1) > 0))[0]
    return float(np.mean([ndcg_score(Yt[i:i+1], Pp[i:i+1], k=k) for i in keep])), len(keep), G, samp["warmth"].median()

for reg, (lo, hi) in {"весь поток": (0.0, 1.0), "train 0-70%": (0.0, 0.7), "test 85-100%": (0.85, 1.0)}.items():
    nd, nn, G, wmed = ma_eval_region("tgbn-genre", lo, hi)
    print(f"genre [{reg:>14}]: MA NDCG@10={nd:.3f}  (n={nn}, classes={G}, med.тёплость={wmed:.0f})")

## Контроль: эвристика на холодных НЕ страдает → cold-gap специфичен для модели

MA-по-истории даёт почти плоскую кривую по тёплости на обоих датасетах (trade: 0.847→0.811, даже слегка падает из-за дрейфа; genre: ~0.84 ровно). То есть **холодным юзерам персональный сигнал доступен и эвристике хватает уже пары прошлых дней.** Слабость на холодных из P2 (TGNv2 trade: 0.378→0.641) — это **cold-start обучаемой памяти/GNN**, а не свойство данных.

> ⚠️ **Дисклеймер по калибровке.** Абсолютные NDCG MA-из-CSV **не сопоставимы со статьёй**: на trade совпало (0.83 ≈ Table 1 0.823), на genre — нет (**0.85 vs 0.509**). Диагностика ниже исключила время-сплит (test-регион тоже 0.84) и число классов (513). Причина — мой MA считается напрямую по идеализированным per-user распределениям из CSV («предскажи своё сглаженное прошлое»), это проще официального пайплайна TGB (edge-stream → память → MovAvg по сообщениям + Evaluator + тестовый набор узлов). Поэтому из MA-секции берём только **форму кривой** (плоская по тёплости), а не абсолютные значения и не её «oracle-uplift». Достоверный headroom считаем на **обученной модели** (ниже).

In [4]:
import sys, torch
REPO = "/Users/aleksandrpanysev/Documents/GitHub/2Q_2026_tgn_user_item"
if REPO not in sys.path: sys.path.insert(0, REPO)
from models.mtgn import MTGNMemory, LastAggregator, LastNeighborLoader
from models.embmodule import MGraphAttentionEmbedding
from models.msgmodule import EncodeIndexModule
from models.decoder import NodePredictor
from tgb.nodeproppred.dataset_pyg import PyGNodePropPredDataset
from torch_geometric.loader import TemporalDataLoader

device = torch.device("cpu"); torch.manual_seed(1)
DIM, NBR, EPOCHS, BS = 128, 10, 10, 200
dataset = PyGNodePropPredDataset(name="tgbn-trade", root="datasets")
data = dataset.get_TemporalData().to(device)
num_classes, raw = dataset.num_classes, data.msg.size(-1)
tl = TemporalDataLoader(data[dataset.train_mask], batch_size=BS)
vl = TemporalDataLoader(data[dataset.val_mask], batch_size=BS)
el = TemporalDataLoader(data[dataset.test_mask], batch_size=BS)
neighbor_loader = LastNeighborLoader(data.num_nodes, size=NBR, device=device)
mm = EncodeIndexModule(DIM, raw, DIM, DIM)
memory = MTGNMemory(data.num_nodes, raw, DIM, DIM, DIM, message_module=mm,
                    aggregator_module=LastAggregator(mm.out_channels)).to(device)
gnn = MGraphAttentionEmbedding(in_channels=DIM, out_channels=DIM, msg_dim=raw, time_enc=memory.time_enc).to(device).float()
node_pred = NodePredictor(in_dim=DIM, out_dim=num_classes).to(device)
opt = torch.optim.Adam(set(memory.parameters())|set(gnn.parameters())|set(node_pred.parameters()), lr=1e-3)
assoc = torch.empty(data.num_nodes, dtype=torch.long, device=device)

@torch.no_grad()
def capture(loader):
    memory.eval(); gnn.eval(); node_pred.eval()
    out, label_t = [], dataset.get_label_time()
    for batch in loader:
        src, dst, t, msg = batch.src, batch.dst, batch.t, batch.msg
        if batch.t[-1] > label_t:
            lt = dataset.get_node_label(batch.t[-1])
            if lt is None: break
            _, lsrc, labels = lt; label_t = dataset.get_label_time()
            pm = batch.t < label_t
            if src[pm].nelement()>0: memory.update_state(src[pm],dst[pm],t[pm],msg[pm]); neighbor_loader.insert(src[pm],dst[pm])
            src,dst,t,msg = src[~pm],dst[~pm],t[~pm],msg[~pm]
            nidn,ei,eid = neighbor_loader(lsrc); assoc[nidn]=torch.arange(nidn.size(0))
            z,lu = memory(nidn); z = gnn(z,lu,ei,data.t[eid],data.msg[eid])
            out.append({"users": lsrc.cpu().numpy(), "y_true": labels.cpu().numpy(),
                        "y_pred": node_pred(z[assoc[lsrc]]).cpu().numpy()})
        if src.nelement()>0: memory.update_state(src,dst,t,msg); neighbor_loader.insert(src,dst)
    return out

crit = torch.nn.CrossEntropyLoss()
for epoch in range(1, EPOCHS+1):
    memory.train(); gnn.train(); node_pred.train(); memory.reset_state(); neighbor_loader.reset_state()
    label_t = dataset.get_label_time()
    for batch in tl:
        opt.zero_grad(); src,dst,t,msg = batch.src,batch.dst,batch.t,batch.msg
        if batch.t[-1] > label_t:
            _, lsrc, labels = dataset.get_node_label(batch.t[-1]); label_t = dataset.get_label_time()
            pm = batch.t < label_t
            if src[pm].nelement()>0: memory.update_state(src[pm],dst[pm],t[pm],msg[pm]); neighbor_loader.insert(src[pm],dst[pm])
            src,dst,t,msg = src[~pm],dst[~pm],t[~pm],msg[~pm]
            nidn,ei,eid = neighbor_loader(lsrc); assoc[nidn]=torch.arange(nidn.size(0))
            z,lu = memory(nidn); z = gnn(z,lu,ei,data.t[eid],data.msg[eid])
            crit(node_pred(z[assoc[lsrc]]), labels).backward(); opt.step()
        if src.nelement()>0: memory.update_state(src,dst,t,msg); neighbor_loader.insert(src,dst)
        memory.detach()
    if epoch < EPOCHS: dataset.reset_label_time()
m_val = capture(vl); m_test = capture(el); dataset.reset_label_time()
print(f"trade TGNv2 обучен. test дней={len(m_test)}, (user,день)={sum(len(b['users']) for b in m_test)}")

trade TGNv2 обучен. test дней=3, (user,день)=665


In [6]:
# Модельный headroom на trade (реальный пайплайн, сопоставим с P0/статьёй)
def model_df(out, train_src):
    hist = (pl.DataFrame({"user": np.asarray(train_src)}).group_by("user")
              .agg(pl.len().alias("warmth")))
    rows = []
    for blk in out:
        yt, yp, us = blk["y_true"], blk["y_pred"], blk["users"]
        for i in range(len(us)):
            if yt[i].sum() > 0:
                rows.append((int(us[i]), float(ndcg_score(yt[i:i+1], yp[i:i+1], k=10))))
    df = pl.DataFrame(rows, schema=["user", "ndcg"], orient="row")
    return df.join(hist, on="user", how="left").with_columns(pl.col("warmth").fill_null(0))

df_model = model_df(m_test, data.src[dataset.train_mask].cpu().numpy())
print(f"trade TGNv2 test NDCG@10 = {df_model['ndcg'].mean():.3f}  (P0 d=256/20эп: 0.646; paper full: 0.735)\n")
res_trade_model = headroom(df_model, "tgbn-trade · TGNv2 (модель)")

trade TGNv2 test NDCG@10 = 0.511  (P0 d=256/20эп: 0.646; paper full: 0.735)

=== tgbn-trade · TGNv2 (модель) ===
shape: (5, 4)
┌────────────────────────────┬────────────┬───────┬─────┐
│ bin                        ┆ avg_warmth ┆ ndcg  ┆ n   │
│ ---                        ┆ ---        ┆ ---   ┆ --- │
│ cat                        ┆ f64        ┆ f64   ┆ u32 │
╞════════════════════════════╪════════════╪═══════╪═════╡
│ (-inf, 328.8000000000002]  ┆ 152.5      ┆ 0.384 ┆ 133 │
│ (328.8000000000002, 757.6] ┆ 586.5      ┆ 0.44  ┆ 133 │
│ (757.6, 1494]              ┆ 1088.8     ┆ 0.535 ┆ 135 │
│ (1494, 2554]               ┆ 1956.4     ┆ 0.53  ┆ 132 │
│ (2554, inf]                ┆ 3641.2     ┆ 0.666 ┆ 132 │
└────────────────────────────┴────────────┴───────┴─────┘
тёплая когорта W = 0.666 | текущий общий NDCG@10 = 0.511
oracle (все холодные → уровень тёплых) = 0.715
UPLIFT = +0.204  (39.8% относительно; закрывает 42% оставшегося до 1.0)



## genre · модельный headroom (из дампа обученного TGNv2)

Фоновый прогон TGNv2 на genre (5 эпох, d=128, `--dump_eval_preds`) дал test NDCG@10 = **0.461** (≈ paper 0.469). Загружаем дамп per-(user, день) NDCG и считаем headroom тем же `headroom()`, что и для trade. Тёплость = число train-рёбер юзера (те же целочисленные node id, что в дампе).

In [7]:
# дамп обученного TGNv2 на genre (ts, user, ndcg)
g_preds = pl.read_parquet(f"{REPO}/mlruns/_eval_preds_tgbn-genre.parquet")
# warmth = число train-рёбер юзера (node id совпадают с дампом)
gds = PyGNodePropPredDataset(name="tgbn-genre", root="datasets")
gdata = gds.get_TemporalData()
g_hist = (pl.DataFrame({"user": gdata.src[gds.train_mask].cpu().numpy()})
            .group_by("user").agg(pl.len().alias("warmth")))
g_df = g_preds.join(g_hist, on="user", how="left").with_columns(pl.col("warmth").fill_null(0))
print(f"genre TGNv2 test NDCG@10 = {g_df['ndcg'].mean():.3f}  (paper full: 0.469) | строк={g_df.height}")
res_genre_model = headroom(g_df, "tgbn-genre · TGNv2 (модель)")

genre TGNv2 test NDCG@10 = 0.465  (paper full: 0.469) | строк=32151
=== tgbn-genre · TGNv2 (модель) ===
shape: (5, 4)
┌────────────────┬────────────┬───────┬──────┐
│ bin            ┆ avg_warmth ┆ ndcg  ┆ n    │
│ ---            ┆ ---        ┆ ---   ┆ ---  │
│ cat            ┆ f64        ┆ f64   ┆ u32  │
╞════════════════╪════════════╪═══════╪══════╡
│ (-inf, 209]    ┆ 4.1        ┆ 0.455 ┆ 6452 │
│ (209, 7832]    ┆ 4050.0     ┆ 0.407 ┆ 6411 │
│ (7832, 18452]  ┆ 12676.7    ┆ 0.446 ┆ 6539 │
│ (18452, 35833] ┆ 26232.6    ┆ 0.472 ┆ 6381 │
│ (35833, inf]   ┆ 62722.7    ┆ 0.544 ┆ 6368 │
└────────────────┴────────────┴───────┴──────┘
тёплая когорта W = 0.544 | текущий общий NDCG@10 = 0.465
oracle (все холодные → уровень тёплых) = 0.625
UPLIFT = +0.161  (34.6% относительно; закрывает 30% оставшегося до 1.0)



## Вывод: headroom от холодных/слабых юзеров

| датасет | TGNv2 test | слабейшая→тёплая | uplift (всех → уровень тёплых) | отн. | закрывает остатка |
|---|---|---|---|---|---|
| tgbn-trade | 0.511 | 0.384 → 0.666 (монотонно) | 0.511 → **0.715** | **+40%** | 42% |
| tgbn-genre | 0.465 | 0.407 → 0.544 (**не монотонно**) | 0.465 → **0.625** | **+35%** | 30% |

**Ответ на вопрос.** Если довести слабые когорты до уровня самых тёплых, общий NDCG@10 растёт существенно на обоих: **trade +0.204, genre +0.161**. Слабая когорта — большая доля ошибки.

**Важная разница в форме.**
- **trade** — чисто монотонно: холодные хуже всего (0.384) → cold-start памяти, как в P2.
- **genre** — **не монотонно**: холодные (warmth≈4) дают 0.455, а *хуже всех* — средне-тёплые (≈4050 рёбер, 0.407); самые тёплые (≈63k) — 0.544, и это уже **выше** лучшей эвристики статьи (MovAvg(L) genre = 0.509). Каждый бин ~6400 точек, провал значим. → На genre мишень не «холодные», а «недотянутые средние»; самые data-rich юзеры задают высокую планку.

**Калибровка/оговорки.** genre-модель near-paper (0.465 ≈ 0.469) → genre-uplift хорошо откалиброван. trade-модель недообучена (0.511 vs 0.735) → trade-uplift скорее верхняя оценка. Тёплость = число train-рёбер юзера.

**Связь с методом.** На обоих датасетах есть большой запас (+0.16…+0.20) от подтягивания слабых когорт к сильным. Плотная per-(user, item) память даёт персональную память сразу — целит в cold-start (trade) и в недотянутые средние когорты (genre), где самые тёплые юзеры уже бьют эвристику.